<a href="https://colab.research.google.com/github/BalytskyiJaroslaw/BioMedAgent_Adapted_for_Google_Colab/blob/main/BioMedAgent_Download_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
import re
import json
import hashlib
import requests
import subprocess
from pathlib import Path

# =========================
# CONFIG
# =========================
ROOT = "/content/drive/MyDrive/BioMedAgent/HuggingFaceData"
TMP_DIR = "/content/zenodo_tmp"
ZENODO_RECORD = "17430550"

os.makedirs(ROOT, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)

print("ROOT   =", ROOT)
print("TMP_DIR=", TMP_DIR)

# =========================
# HELPERS
# =========================
def run(cmd):
    print("RUN:", " ".join(cmd))
    subprocess.run(cmd, check=True)

def md5sum(path, chunk_size=1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

# =========================
# Ensure unzip exists
# =========================
try:
    run(["unzip", "-v"])
except Exception:
    run(["apt-get", "-qq", "update"])
    run(["apt-get", "-qq", "install", "-y", "unzip"])

# =========================
# Query Zenodo
# =========================
api_url = f"https://zenodo.org/api/records/{ZENODO_RECORD}"
r = requests.get(api_url, timeout=60)
r.raise_for_status()
record = r.json()

parts = []
for f in record.get("files", []):
    name = f.get("key", "")
    url = f.get("links", {}).get("self", "")
    size = f.get("size", 0)
    checksum = f.get("checksum", "")  # e.g. md5:abcdef...

    if re.match(r"biomedical_dataset_part\d{2}_of_30\.zip$", name):
        md5 = checksum.split("md5:")[-1] if "md5:" in checksum else None
        parts.append({
            "name": name,
            "url": url,
            "size": size,
            "md5": md5,
        })

parts = sorted(parts, key=lambda x: x["name"])
print(f"Found {len(parts)} parts")
print("Total ZIP size (GB):", round(sum(p['size'] for p in parts) / (1024**3), 2))

# Save manifest for later reference
with open(os.path.join(ROOT, "zenodo_manifest.json"), "w") as f:
    json.dump(parts, f, indent=2)

# =========================
# Download/extract loop
# =========================
for i, part in enumerate(parts, start=1):
    name = part["name"]
    url = part["url"]
    expected_md5 = part["md5"]

    zip_path = os.path.join(TMP_DIR, name)
    done_marker = os.path.join(ROOT, f".{name}.done")

    if os.path.exists(done_marker):
        print(f"[{i}/{len(parts)}] SKIP already done: {name}")
        continue

    print(f"\n[{i}/{len(parts)}] Processing {name}")

    # Download with resume support
    run(["wget", "-c", "-O", zip_path, url])

    # Verify MD5 if available
    if expected_md5:
        actual_md5 = md5sum(zip_path)
        print("Expected MD5:", expected_md5)
        print("Actual   MD5:", actual_md5)
        if actual_md5.lower() != expected_md5.lower():
            raise RuntimeError(f"MD5 mismatch for {name}")

    # Extract directly into Drive
    run(["unzip", "-o", zip_path, "-d", ROOT])

    # Mark done and delete zip
    Path(done_marker).write_text("done\n")
    os.remove(zip_path)

    print(f"[{i}/{len(parts)}] Finished {name}")

print("\nALL AVAILABLE PARTS PROCESSED.")

downloaded_root = os.path.join(ROOT, "downloaded_files")
print("downloaded_files exists:", os.path.exists(downloaded_root))
if os.path.exists(downloaded_root):
    entries = sorted(os.listdir(downloaded_root))
    print("Number of extracted index folders:", len(entries))
    print("First 10:", entries[:10])
    print("Last 10 :", entries[-10:])

ROOT   = /content/drive/MyDrive/BioMedAgent/HuggingFaceData
TMP_DIR= /content/zenodo_tmp
RUN: unzip -v
Found 30 parts
Total ZIP size (GB): 29.01

[1/30] Processing biomedical_dataset_part01_of_30.zip
RUN: wget -c -O /content/zenodo_tmp/biomedical_dataset_part01_of_30.zip https://zenodo.org/api/records/17430550/files/biomedical_dataset_part01_of_30.zip/content
Expected MD5: ac5f80473e6f55aabe49b4049bbe4e7a
Actual   MD5: ac5f80473e6f55aabe49b4049bbe4e7a
RUN: unzip -o /content/zenodo_tmp/biomedical_dataset_part01_of_30.zip -d /content/drive/MyDrive/BioMedAgent/HuggingFaceData
[1/30] Finished biomedical_dataset_part01_of_30.zip

[2/30] Processing biomedical_dataset_part02_of_30.zip
RUN: wget -c -O /content/zenodo_tmp/biomedical_dataset_part02_of_30.zip https://zenodo.org/api/records/17430550/files/biomedical_dataset_part02_of_30.zip/content
Expected MD5: 0c1d4b09c1300e4d1495891161f97be6
Actual   MD5: 0c1d4b09c1300e4d1495891161f97be6
RUN: unzip -o /content/zenodo_tmp/biomedical_dataset_part

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/BioMedAgent/HuggingFaceData")

# show suspicious top-level entries
items = sorted(ROOT.iterdir(), key=lambda p: p.name)
print("Top-level entries:", len(items))
for p in items[:50]:
    print(p.name)

print("\nEntries containing backslashes:")
bad = [p for p in items if "\\" in p.name]
print("Count:", len(bad))
for p in bad[:30]:
    print(p.name)

Top-level entries: 364
.biomedical_dataset_part01_of_30.zip.done
.biomedical_dataset_part02_of_30.zip.done
.biomedical_dataset_part03_of_30.zip.done
.biomedical_dataset_part04_of_30.zip.done
.biomedical_dataset_part05_of_30.zip.done
.biomedical_dataset_part06_of_30.zip.done
.biomedical_dataset_part07_of_30.zip.done
.biomedical_dataset_part08_of_30.zip.done
.biomedical_dataset_part09_of_30.zip.done
.biomedical_dataset_part10_of_30.zip.done
.biomedical_dataset_part11_of_30.zip.done
.biomedical_dataset_part12_of_30.zip.done
.biomedical_dataset_part13_of_30.zip.done
.biomedical_dataset_part14_of_30.zip.done
.biomedical_dataset_part15_of_30.zip.done
.biomedical_dataset_part16_of_30.zip.done
.biomedical_dataset_part17_of_30.zip.done
.biomedical_dataset_part18_of_30.zip.done
.biomedical_dataset_part19_of_30.zip.done
.biomedical_dataset_part20_of_30.zip.done
.biomedical_dataset_part21_of_30.zip.done
.biomedical_dataset_part22_of_30.zip.done
.biomedical_dataset_part23_of_30.zip.done
.biomedical

In [ ]:
from pathlib import Path
import os

ROOT = Path("/content/drive/MyDrive/BioMedAgent/HuggingFaceData")

entries = sorted([p.name for p in ROOT.iterdir() if p.is_dir() and p.name.startswith("index_")])
print("Number of index folders:", len(entries))
print("First 10:", entries[:10])
print("Last 10 :", entries[-10:])

Number of index folders: 327
First 10: ['index_0001', 'index_0002', 'index_0003', 'index_0004', 'index_0005', 'index_0006', 'index_0007', 'index_0008', 'index_0009', 'index_0010']
Last 10 : ['index_0318', 'index_0319', 'index_0320', 'index_0321', 'index_0322', 'index_0323', 'index_0324', 'index_0325', 'index_0326', 'index_0327']


In [ ]:
from pathlib import Path
import os

ROOT = Path("/content/drive/MyDrive/BioMedAgent/HuggingFaceData")
LIST_FILE = ROOT / "dataset_files_list.txt"

print("=" * 100)
print("ROOT:", ROOT)
print("ROOT exists:", ROOT.exists())
print("=" * 100)

# --------------------------------------------------
# 1) Find all extracted index folders
# --------------------------------------------------
index_dirs = sorted([p for p in ROOT.iterdir() if p.is_dir() and p.name.startswith("index_")])
index_names = [p.name for p in index_dirs]

print(f"Index folders found: {len(index_dirs)}")
print("First 10 index dirs:", index_names[:10])
print("Last 10 index dirs :", index_names[-10:])

expected_index_names = [f"index_{i:04d}" for i in range(1, 328)]
missing_index_dirs = [x for x in expected_index_names if x not in index_names]
extra_index_dirs = [x for x in index_names if x not in expected_index_names]

print("\nMissing index dirs:", len(missing_index_dirs))
if missing_index_dirs[:20]:
    print("First missing:", missing_index_dirs[:20])

print("Extra index dirs:", len(extra_index_dirs))
if extra_index_dirs[:20]:
    print("Extra:", extra_index_dirs[:20])

# --------------------------------------------------
# 2) Count files and total size only inside index_ dirs
# --------------------------------------------------
actual_files = []
total_bytes = 0

for idx in index_dirs:
    for p in idx.rglob("*"):
        if p.is_file():
            rel = p.relative_to(ROOT).as_posix()
            size = p.stat().st_size
            actual_files.append((rel, size))
            total_bytes += size

actual_relpaths = set(rel for rel, _ in actual_files)

print("\n" + "=" * 100)
print("EXTRACTED PAYLOAD SUMMARY")
print("=" * 100)
print("Actual extracted files:", len(actual_files))
print("Actual total size (GB):", round(total_bytes / (1024**3), 2))

# --------------------------------------------------
# 3) Parse expected file list from dataset_files_list.txt
#    Normalize:
#    downloaded_files\index_0001\files\a.txt
#    -> index_0001/files/a.txt
# --------------------------------------------------
expected_relpaths = set()

if LIST_FILE.exists():
    with open(LIST_FILE, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            line = line.replace("\\", "/")
            if line.startswith("downloaded_files/"):
                line = line[len("downloaded_files/"):]
            expected_relpaths.add(line)

print("\nExpected files from dataset_files_list.txt:", len(expected_relpaths))

missing_files = sorted(expected_relpaths - actual_relpaths)
extra_files = sorted(actual_relpaths - expected_relpaths)

print("Missing files vs expected list:", len(missing_files))
if missing_files[:20]:
    print("First missing files:")
    for x in missing_files[:20]:
        print("  ", x)

print("\nExtra files vs expected list:", len(extra_files))
if extra_files[:20]:
    print("First extra files:")
    for x in extra_files[:20]:
        print("  ", x)

# --------------------------------------------------
# 4) Show a few sample actual files
# --------------------------------------------------
print("\n" + "=" * 100)
print("SAMPLE EXTRACTED FILES")
print("=" * 100)
for rel, size in actual_files[:20]:
    print(f"{rel}   [{round(size / (1024**2), 3)} MB]")

# --------------------------------------------------
# 5) Largest files
# --------------------------------------------------
largest = sorted(actual_files, key=lambda x: x[1], reverse=True)[:20]

print("\n" + "=" * 100)
print("TOP 20 LARGEST FILES")
print("=" * 100)
for rel, size in largest:
    print(f"{round(size / (1024**3), 3):>8} GB   {rel}")

# --------------------------------------------------
# 6) Include metadata size too (whole ROOT)
# --------------------------------------------------
root_total_bytes = 0
root_total_files = 0
for p in ROOT.rglob("*"):
    if p.is_file():
        root_total_files += 1
        root_total_bytes += p.stat().st_size

print("\n" + "=" * 100)
print("WHOLE ROOT SUMMARY (payload + metadata)")
print("=" * 100)
print("Total files under ROOT:", root_total_files)
print("Total size under ROOT (GB):", round(root_total_bytes / (1024**3), 2))

# --------------------------------------------------
# 7) Quick verdict
# --------------------------------------------------
print("\n" + "=" * 100)
print("VERDICT")
print("=" * 100)

ok_dirs = (len(index_dirs) == 327 and len(missing_index_dirs) == 0)
ok_files = (len(missing_files) == 0)

if ok_dirs and ok_files:
    print("Looks structurally correct.")
else:
    print("Something is still off structurally.")

print(f"Expected ~31.1 GB from README; actual extracted payload = {round(total_bytes / (1024**3), 2)} GB")

ROOT: /content/drive/MyDrive/BioMedAgent/HuggingFaceData
ROOT exists: True
Index folders found: 327
First 10 index dirs: ['index_0001', 'index_0002', 'index_0003', 'index_0004', 'index_0005', 'index_0006', 'index_0007', 'index_0008', 'index_0009', 'index_0010']
Last 10 index dirs : ['index_0318', 'index_0319', 'index_0320', 'index_0321', 'index_0322', 'index_0323', 'index_0324', 'index_0325', 'index_0326', 'index_0327']

Missing index dirs: 0
Extra index dirs: 0

EXTRACTED PAYLOAD SUMMARY
Actual extracted files: 1028
Actual total size (GB): 29.95

Expected files from dataset_files_list.txt: 1028
Missing files vs expected list: 0

Extra files vs expected list: 0

SAMPLE EXTRACTED FILES
index_0001/files/MUT_edit1.fastq.gz   [0.198 MB]
index_0001/files/MUT_edit2.fastq.gz   [0.19 MB]
index_0001/milestone/tumor.recal.bam   [0.71 MB]
index_0001/milestone/tumor.recal.fil.maf   [0.003 MB]
index_0001/milestone/tumor.recal.vcf   [0.012 MB]
index_0002/files/MUT_edit1.fastq.gz   [0.198 MB]
index_0